# CutMix Clutter

`CutMixClutter` is a CutMix preset for **target-in-clutter** classification.
For each image, a rectangular patch from another image (`x.flip(0)`) is pasted
in — that patch is the **target** ("inset"), and the surrounding untouched
image is **clutter**.

Unlike standard CutMix, the label is **not** an area-weighted blend: it is the
one-hot **target** class. The clutter class gets no special weight — only the
optional label-smoothing floor, exactly like any other non-target class.

The rectangular patch boundary is always a learnable cue for which region is
the target, so the inset stays discoverable across sizes. `cutmix_minmax`
bounds the inset so it is a genuine sub-region (with some size variability)
rather than covering the whole image.

Use this notebook to eyeball whether the defaults (`cutmix_minmax=(0.3, 0.6)`)
look right — then we can adjust.

In [ ]:
LITDATA_VAL_PATH = "s3://visionlab-datasets/imagenet1k/pre-processed/s256-l512-jpgbytes-q100-streaming/val/"

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from slipstream import SlipstreamDataset, SlipstreamLoader, DecodeRandomResizedCrop
from slipstream.transforms import CutMixClutter, ToFloatDiv

dataset = SlipstreamDataset(remote_dir=LITDATA_VAL_PATH, decode_images=False)
print(f"Dataset: {len(dataset):,} samples")

In [ ]:
def load_batch(dataset, size=224, batch_size=8):
    """Load one batch as float-in-[0,1] images plus int64 labels."""
    dec = DecodeRandomResizedCrop(size=size, to_tensor=True, permute=True)
    loader = SlipstreamLoader(
        dataset, batch_size=batch_size, shuffle=False,
        pipelines={'image': [dec]}, verbose=False,
    )
    batch = next(iter(loader))
    loader.shutdown()
    return {
        'image': batch['image'].float() / 255.0,
        'label': batch['label'].long(),
    }


def show_grid(rows, row_labels=None, col_titles=None, suptitle=None, max_cols=8):
    """Show one or more rows of [B, C, H, W] tensors."""
    if isinstance(rows, torch.Tensor):
        rows = [rows]
    n_rows = len(rows)
    n_cols = min(max_cols, rows[0].shape[0])
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.0 * n_cols, 2.4 * n_rows))
    axes = np.atleast_2d(axes)
    for r, row in enumerate(rows):
        for c in range(n_cols):
            img = row[c].detach().float().permute(1, 2, 0).clamp(0, 1).cpu().numpy()
            axes[r, c].imshow(img)
            axes[r, c].axis('off')
            if r == 0 and col_titles is not None and c < len(col_titles):
                axes[r, c].set_title(col_titles[c], fontsize=9)
        if row_labels:
            axes[r, 0].set_title(row_labels[r], fontsize=10, loc='left', x=-0.05, y=0.4)
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


batch = load_batch(dataset, size=224, batch_size=8)
print(f"image: {tuple(batch['image'].shape)}, label: {tuple(batch['label'].shape)}")

## Default CutMixClutter

Three rows: the **clutter source** (the untouched background image), the
**target source** (`x.flip(0)` — the inset is a bbox crop of this), and the
**cutmix output**. Column titles show the resulting label (= target class)
and the inset size in pixels.

In [ ]:
CutMixClutter?

In [ ]:
clutter = batch['image']
target_src = clutter.flip(0)   # the inset is a bbox crop of this

m = CutMixClutter(num_classes=1000, seed=2026)
out = m({'image': batch['image'].clone(), 'label': batch['label'].clone()})

target_cls = out['label'].argmax(dim=-1).tolist()
yl, yh, xl, xh = m.last_bbox
col_titles = [
    f"label c={target_cls[i]}\ninset {int(xh[i]-xl[i])}x{int(yh[i]-yl[i])} px"
    for i in range(8)
]
show_grid(
    [clutter, target_src, out['image']],
    row_labels=['clutter source', 'target source', 'cutmix output'],
    col_titles=col_titles,
    suptitle='CutMixClutter defaults — inset (bbox from target source) embedded in clutter',
)
print(repr(m))

## The returned label is the target, not a blend

Standard CutMix would weight the label by area. Here the label is the pure
one-hot **inset/target** class; the **clutter** class is treated like any
other non-target class. With `label_smoothing`, every non-target class —
clutter included — gets exactly `smoothing / num_classes`.

(Small `num_classes` here just for a readable bar plot.)

In [ ]:
K = 20  # small for a readable bar plot
small_labels = torch.randint(0, K, (8,))

m0 = CutMixClutter(num_classes=K, label_smoothing=0.0, seed=7)
out0 = m0({'image': batch['image'].clone(), 'label': small_labels.clone()})
m1 = CutMixClutter(num_classes=K, label_smoothing=0.1, seed=7)
out1 = m1({'image': batch['image'].clone(), 'label': small_labels.clone()})

i = 0
target_c = int(small_labels.flip(0)[i])
clutter_c = int(small_labels[i])
print(f"sample {i}: target (inset) class = {target_c}, clutter class = {clutter_c}")
print(f"  smoothing=0.0  -> target weight={out0['label'][i, target_c]:.3f}, "
      f"clutter weight={out0['label'][i, clutter_c]:.3f}")
print(f"  smoothing=0.1  -> target weight={out1['label'][i, target_c]:.3f}, "
      f"clutter weight={out1['label'][i, clutter_c]:.4f}  (= smoothing/K = {0.1 / K:.4f})")

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for ax, lbl, tag in [(axes[0], out0['label'][i], 'smoothing=0.0'),
                     (axes[1], out1['label'][i], 'smoothing=0.1')]:
    bars = ax.bar(np.arange(K), lbl.numpy())
    bars[target_c].set_color('tab:red')
    if clutter_c != target_c:
        bars[clutter_c].set_color('tab:orange')
    ax.set_title(f"sample {i}, {tag}   (red = target, orange = clutter)")
    ax.set_ylabel('probability')
    ax.set_ylim(0, 1.05)
axes[1].set_xlabel('class index')
plt.tight_layout(); plt.show()

## Inset-size sweep — choosing `cutmix_minmax`

`cutmix_minmax` bounds each bbox side as a fraction of the image. This is the
knob that decides how much of the frame is target vs clutter. Eyeball these
to settle on a default.

In [ ]:
ranges = [(0.2, 0.4), (0.3, 0.6), (0.4, 0.8)]
rows, labels = [], []
for rng in ranges:
    ms = CutMixClutter(num_classes=1000, cutmix_minmax=rng, seed=2026)
    o = ms({'image': batch['image'].clone(), 'label': batch['label'].clone()})
    rows.append(o['image'])
    labels.append(f"minmax={rng}")
show_grid(rows, row_labels=labels,
          suptitle='Inset-size sweep — bigger range = larger / more variable target')

## End-to-end via `SlipstreamLoader.after_batch_transforms`

`CutMixClutter` plugs into the loader exactly like `Mixup` — it runs on the
assembled batch dict after the per-field pipelines.

In [ ]:
loader = SlipstreamLoader(
    dataset, batch_size=8, shuffle=False,
    pipelines={'image': [
        DecodeRandomResizedCrop(size=224, to_tensor=True, permute=True),
        ToFloatDiv(255.0),
    ]},
    after_batch_transforms=[CutMixClutter(num_classes=1000, seed=2026)],
    verbose=False,
)
batch_out = next(iter(loader))
loader.shutdown()

print(f"image: {tuple(batch_out['image'].shape)}")
print(f"label: {tuple(batch_out['label'].shape)}  dtype={batch_out['label'].dtype}")
print(f"label sums (should be ~1): {[round(v, 3) for v in batch_out['label'].sum(dim=-1).tolist()]}")
print(f"max per sample (one-hot -> ~1): "
      f"{[round(v, 3) for v in batch_out['label'].max(dim=-1).values.tolist()]}")
show_grid(batch_out['image'], suptitle='CutMixClutter via after_batch_transforms')